<!-- AI-GENERATED
     Model   : Claude Sonnet 4.6
     Date    : 2026-05-08
     Prompt  : can you do this notebook in the expirements uivenus folder with the name runner_on_kaggle
-->

# UI-Venus ScreenSpot-Pro Runner (Kaggle)

This notebook runs full grounding and evaluation for UI-Venus on ScreenSpot-Pro.
Outputs are saved inside this experiment folder.

In [ ]:
%pip install -q pydantic pillow pyyaml loguru huggingface-hub transformers qwen-vl-utils accelerate

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

KAGGLE_REPO = Path('/kaggle/working/Click2Act')
REPO_URL = 'https://github.com/YaraHisham61/Click2Act.git'
REPO_BRANCH = 'uivenus'

if not KAGGLE_REPO.exists():
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, str(KAGGLE_REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(KAGGLE_REPO), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(KAGGLE_REPO), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(KAGGLE_REPO), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)

ROOT = KAGGLE_REPO if KAGGLE_REPO.exists() else Path.cwd().parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Working directory:', ROOT)

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('cuda name:', torch.cuda.get_device_name(0))
    print('compute capability:', torch.cuda.get_device_capability(0))

In [ ]:
GROUNDER_CFG = 'experiments/2026-04-27_exp_uivenus_screenspotpro/grounder.yaml'
EVAL_CFG = 'experiments/2026-04-27_exp_uivenus_screenspotpro/evaluate_grounder.yaml'

print('Grounder config:', GROUNDER_CFG)
print('Eval config:', EVAL_CFG)

In [ ]:
# Full run (resume-safe).
# It saves checkpoints every 300 rows based on save_every_n in grounder.yaml.
!python src/pipeline/grounder.py experiments/2026-04-27_exp_uivenus_screenspotpro/grounder.yaml

In [ ]:
# Evaluation excludes failed/timeout rows from metrics.
!python experiments/2026-04-27_exp_uivenus_screenspotpro/run_evaluation.py

In [ ]:
import json
from pathlib import Path

summary_path = Path('experiments/2026-04-27_exp_uivenus_screenspotpro/eval_summary.json')
valid_csv_path = Path('experiments/2026-04-27_exp_uivenus_screenspotpro/grounder_valid_only.csv')
grounder_csv_path = Path('experiments/2026-04-27_exp_uivenus_screenspotpro/grounder.csv')

print('grounder.csv exists:', grounder_csv_path.exists())
print('grounder_valid_only.csv exists:', valid_csv_path.exists())
print('eval_summary.json exists:', summary_path.exists())

if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))